<a href="https://colab.research.google.com/github/DJean09/2XKO-Nuzlocke-Interactive-Stream-Overlay/blob/main/TravelProReviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**There's 4 code sections below.** The first is just an initializer so it **MUST be ran first**. The other 3 are labeled for each store so you don't have to run through all of them at once.

In [ ]:
# ******************* RUN THIS FIRST *******************
# Imports and Functions

import requests # For making API requests
from bs4 import BeautifulSoup # For parsing HTML content
import json # To parse JSON responses
import time # For rate limiting
from datetime import date # For date formatting

INCLUDE_UNPUBLISHED = False

PER_PAGE_BOTTOMLINES = 100      # max 100
PER_PAGE_WIDGET = 150           # max 150 for product widget
REQUEST_TIMEOUT = 30
SLEEP_BETWEEN_CALLS = 0.15      # be polite to the CDN

# def fetch_all_reviews(key, product_id, per_page=150):
#   """"
#   Fetch all reviews for a given product ID from the Yotpo API.
#   Args:
#     key (str): The Yotpo API key.
#     product_id (str): The product ID to fetch reviews for.
#     per_page (int): Number of reviews to fetch per page. Yotpo allows a maximum of 150.
#   """
#   # Set the base URL for the Yotpo API
#   all_reviews = []
#   page = 1

#   # Loop to fetch all pages of reviews
#   while True:
#     url = f"https://api-cdn.yotpo.com/v1/widget/{key}/products/{product_id}/reviews.json"
#     params = {"per_page": per_page, "page": page}
#     resp = requests.get(url, params=params)
#     resp.raise_for_status()
#     data = resp.json().get("response", {})
#     reviews = data.get("reviews", [])
#     if not reviews:
#       break
#     all_reviews.extend(reviews)
#     pagination = data.get("pagination", {})
#     total = pagination.get("total", 0)
#     if len(all_reviews) >= total:
#       break
#     page += 1
#     time.sleep(0.2)
#   return all_reviews

# def fetch_all_reviews(app_key, api_secret, product_id=None, per_page=100):
#     """
#     Fetch EVERY review (published + hidden/unpublished) via Yotpo UGC v1.
#     If product_id is provided, results are filtered client-side to that product.

#     Args:
#         app_key (str): Yotpo App Key (aka store/account id).
#         api_secret (str): Yotpo API Secret.
#         product_id (str|int|None): Optional product id to filter to.
#         per_page (int): Reviews per page (Yotpo recommends <=100).
#         include_unpublished (bool): If True, include hidden/unpublished reviews.
#     Returns:
#         list[dict]: All matching reviews.
#     """

#     # 1) Get a server-to-server token (a.k.a. utoken / access_token)
#     tok_resp = requests.post(
#         "https://api.yotpo.com/oauth/token",
#         json={
#             "client_id": app_key,
#             "client_secret": api_secret,
#             "grant_type": "client_credentials",
#         },
#         headers={"accept": "application/json"},
#         timeout=30,
#     )
#     tok_resp.raise_for_status()
#     utoken = tok_resp.json()["access_token"]

#     # 2) Page through the account-level reviews (this is NOT the widget feed)
#     all_reviews, page = [], 1
#     while True:
#         params = {
#             "utoken": utoken,
#             "page": page,
#             "count": per_page,
#         }
#         # Yotpo’s v1 docs list a boolean to include unpublished;
#         # it may appear as `deleted` in the docs UI. Pass it when requested.
#         if INCLUDE_UNPUBLISHED:
#             params["deleted"] = True  # includes/excludes unpublished/hidden in results

#         r = requests.get(
#             f"https://api.yotpo.com/v1/apps/{app_key}/reviews",
#             params=params,
#             headers={"accept": "application/json"},
#             timeout=30,
#         )
#         r.raise_for_status()
#         items = r.json().get("reviews", []) or []
#         if not items:
#             break

#         # Optional product filter (client-side) if you only want one product
#         if product_id is not None:
#             pid_str = str(product_id)
#             items = [rev for rev in items if str(rev.get("product_id")) == pid_str]

#         all_reviews.extend(items)

#         # Stop when we obviously reached the end
#         if len(items) < per_page:
#             break

#         page += 1
#         time.sleep(0.2)

#     return all_reviews

# =========================
# HELPERS
# =========================
def _get(session: requests.Session, url: str, params: dict | None = None) -> dict:
    r = session.get(url, params=params or {}, timeout=REQUEST_TIMEOUT,
                    headers={"accept": "application/json"})
    r.raise_for_status()
    return r.json()

# =========================
# A) Discover product IDs (domain_keys) that have published reviews
# =========================
def discover_product_ids_with_reviews(app_key: str, per_page: int = PER_PAGE_BOTTOMLINES) -> list[str]:
    assert 1 <= per_page <= 100, "bottom_lines allows up to 100 per page."
    session = requests.Session()
    base = f"https://api-cdn.yotpo.com/v1/apps/{app_key}/bottom_lines"

    product_ids: list[str] = []
    page = 1

    while True:
        payload = _get(session, base, params={"count": per_page, "page": page})
        resp = payload.get("response") or {}
        bottoms = resp.get("bottomlines") or []
        if not bottoms:
            break

        for bl in bottoms:
            pid = str(bl.get("domain_key") or "").strip()
            total_reviews = int(bl.get("total_reviews") or 0)
            if pid and total_reviews > 0:
                product_ids.append(pid)

        if len(bottoms) < per_page:
            break

        page += 1
        time.sleep(SLEEP_BETWEEN_CALLS)

    # De-dupe while preserving order
    seen = set()
    return [pid for pid in product_ids if not (pid in seen or seen.add(pid))]

# =========================
# B) Fetch PDP widget reviews (published-only) + product name
# =========================
def _extract_product_name(resp_products, product_id: str) -> str | None:
    """
    Yotpo may return products as:
      - dict: { "<product_id>": { name: ... } }
      - list: [ { id/domain_key/name: ... }, ... ]
    Try both forms; fall back to first available name.
    """
    if not resp_products:
        return None

    # Dict form keyed by product_id
    if isinstance(resp_products, dict):
        entry = resp_products.get(str(product_id))
        if isinstance(entry, dict):
            name = entry.get("name")
            if isinstance(name, str) and name.strip():
                return name.strip()

        # sometimes keys may be ints; try a loose scan
        for v in resp_products.values():
            if isinstance(v, dict) and isinstance(v.get("name"), str) and v["name"].strip():
                return v["name"].strip()

    # List form
    if isinstance(resp_products, list):
        # try to match on id/domain_key first
        for p in resp_products:
            if not isinstance(p, dict):
                continue
            if str(p.get("id")) == str(product_id) or str(p.get("domain_key")) == str(product_id):
                name = p.get("name")
                if isinstance(name, str) and name.strip():
                    return name.strip()
        # else just take the first with a name
        for p in resp_products:
            if isinstance(p, dict) and isinstance(p.get("name"), str) and p["name"].strip():
                return p["name"].strip()

    return None

def fetch_displayed_product_reviews(
    app_key: str,
    product_id: str,
    per_page: int = PER_PAGE_WIDGET,
    sort: str = "date",
    direction: str = "desc",
) -> tuple[list[dict], str | None]:
    """
    Returns (reviews_list, product_name) from the storefront product widget feed.
    GET https://api-cdn.yotpo.com/v1/widget/{app_key}/products/{product_id}/reviews.json
    """
    assert 1 <= per_page <= 150, "product widget allows up to 150 per page."

    session = requests.Session()
    base = f"https://api-cdn.yotpo.com/v1/widget/{app_key}/products/{product_id}/reviews.json"

    all_reviews: list[dict] = []
    product_name: str | None = None
    page = 1

    while True:
        params = {"page": page, "per_page": per_page, "sort": sort, "direction": direction}
        payload = _get(session, base, params=params)
        resp = payload.get("response") or {}

        # Capture product name once
        if product_name is None:
            product_name = _extract_product_name(resp.get("products"), str(product_id))
            # If still None, try first review's product_title as a fallback
            if not product_name:
                first_reviews = resp.get("reviews") or []
                if first_reviews and isinstance(first_reviews, list):
                    maybe = first_reviews[0].get("product_title")
                    if isinstance(maybe, str) and maybe.strip():
                        product_name = maybe.strip()

        # Accumulate reviews
        page_reviews = resp.get("reviews") or []
        if not isinstance(page_reviews, list):
            # be defensive; if the API shape changes, don't crash
            page_reviews = []
        all_reviews.extend(page_reviews)

        pagination = resp.get("pagination") or {}
        total = int(pagination.get("total") or 0)

        if page * per_page >= total or not page_reviews:
            break

        page += 1
        time.sleep(SLEEP_BETWEEN_CALLS)

    return all_reviews, product_name

# =========================
# C) Output formatting — print ONE review at a time (no indexing)
# =========================
def print_review(review: dict, file, product_name: str):
    file.write(f"PRODUCT NAME: {product_name}\n")
    file.write(f"REVIEW ID: {review.get('id')}\n")
    file.write(f"Created At: {review.get('created_at')}\n")
    file.write(f"Review Title: {review.get('title')}\n")
    file.write(f"Written Review: {review.get('content')}\n")
    file.write(f"Review Score: {review.get('score')}\n")
    file.write(f"Sentiment: {review.get('sentiment')}\n")
    file.write("------------------------------------------------------------------\n\n")


In [ ]:
# TravelPro US STORE

# App Key for Yotpo API
# This key is used to authenticate requests to the Yotpo API.
APP_KEY = "w2Y8hdJ6MIDrMECQOQ4g9DbE82K33tOsdxuLNKvh"
API_SECRET = "wJF5LRxRx65CuSMuqoDG1JDtYhZxLDAzCw5GIDEN"

# This dictionary contains product IDs and their corresponding names.
PRODUCTS = {
    '7349346173026': 'Platinum Elite Weekender Bag / Medium Check-In Hardside Set',
    '7350348349538': 'Maxlite 5 Soft Tote / Carry-On Spinner Set',
    '7353153454178': 'Platinum Elite Compact Carry-On / Medium Check-In Hardside Set',
    '7349325103202': 'Crew Classic Carry- On / Medium / Large Set',
    '7350417850466': 'Maxlite 5 Soft Tote / Carry-On Spinner Set',
    '7345931616354': 'Platinum Elite Medium / Large Check-In Set',
    '7353154502754': 'Platinum Elite Compact Carry-On / Medium Check-In Hardside Set',
    '7353156403298': 'Platinum Elite Compact Carry-On / Medium Check-In Hardside Set',
    '7349754167394': 'Platinum Elite Weekender Bag / Medium Check-In Hardside Set',
    '7350352248930': 'Maxlite 5 Soft Tote / Carry-On Spinner Set',
    '7349327069282': 'Crew Classic Carry- On / Medium / Large Set',
    '7349336277090': 'Crew Classic Carry- On / Medium / Large Set',
    '7345938301026': 'Platinum Elite Medium / Large Check-In Set',
    '7345926570082': 'Platinum Elite Medium / Large Check-In Set',
    '7351393583202': 'Platinum Elite Weekender Bag / Medium Check-In Hardside Set',
    '7353155059810': 'Platinum Elite Compact Carry-On / Medium Check-In Hardside Set',
    '7350354215010': 'Maxlite 5 Soft Tote / Carry-On Spinner Set',
    '7345894424674': 'Platinum Elite Medium / Large Check-In Set',
    '7345830133858': 'Platinum Elite Medium / Large Check-In Set',
    '7349267005538': 'Crew Classic Carry- On / Medium / Large Set',
    '7349752266850': 'Platinum Elite Weekender Bag / Medium Check-In Hardside Set',
    '7345890525282': 'Platinum Elite Medium / Large Check-In Set',
    '7330306850914': 'Platinum Elite Carry-On / Business Backpack Set',
    '7321843040354': 'Platinum Elite Carry-On / UnderSeat Tote Camouflage Set',
    '7330425438306': 'Crew Classic Carry-On / Large Check-In Set',
    '7322270007394': 'Platinum Elite Hardside Carry on / Travelpro x Travel + Leisure Weekender Set',
    '7332746297442': 'Maxlite Air V2 Carry-On / Large Check-In Set',
    '7203001106530': 'Roundtrip Carry-on & Medium Check-in Spinner  Set',
    '7332751736930': 'Maxlite Air V2 Carry-On / Large Check-In Set',
    '7323945861218': 'Platinum Elite Medium / Large Check-In Camouflage Set',
    '7332752490594': 'Maxlite Air V2 Carry-On / Medium Check-In Set',
    '7231722324066': 'Platinum Elite Soft Duffel - Camouflage',
    '7332752818274': 'Maxlite Air V2 Carry-On / Medium Check-In Set',
    '7325349904482': 'Travelpro x SURGEON Weekender',
    '7332753440866': 'Maxlite Air V2 Carry-On / Medium / Large Set',
    '7202368847970': 'Platinum Elite Business Backpack',
    '7332757078114': 'Maxlite Air V2 Carry-On / Medium / Large Set',
    '7327858786402': 'Travelpro Altitude Sling 2L',
    '7334495191138': 'Versapack+ Carry-on Spinner / Travelpro Altitude Large Backpack Set',
    '7270779256930': 'Travelpro Platinum Elite Carry-On Spinner - Camouflage',
    '7334916161634': 'Crew Classic Medium Check in Spinner / Travelpro Altitude Large Backpack Set',
    '7327960891490': 'Platinum Elite Carry-On / Large Check-In Hardside Set',
    '7335064535138': 'Platinum Elite Carry-On / UnderSeat Tote Set',
    '7209949888610': 'Travelpro Altitude Large Expandable Laptop Backpack 30-36L',
    '7335941832802': 'Platinum Elite Carry-On / Slim Backpack Set',
    '7327961841762': 'Platinum Elite Carry-On / Large Check-In Hardside Set',
    '7339136254050': 'Crew Classic Compact Carry-On / Medium Check-In Set',
    '7307356438626': 'Maxlite Air V2 Compact Carry-On Hardside Spinner',
    '7339158503522': 'Crew Classic Compact Carry-On / Large Check-In Set',
    '7329627209826': 'Crew Classic Carry-On / Medium Check-In Set',
    '7339982979170': 'Platinum Elite Carry-On / Medium Check-In Hardside Set',
    '7202966765666': 'Roundtrip UnderSeat Soft Tote',
    '7342365737058': 'Platinum Elite Carry-On / Medium Check-In Hardside Set',
    '7329629143138': 'Crew Classic Carry-On / Medium Check-In Set',
    '7343145746530': 'Platinum Elite Carry-On / Medium Check-In Spinner Set',
    '7307874631778': 'Maxlite Air V2 Large Check-In Hardside Spinner',
    '7343154266210': 'Platinum Elite Carry-On / Medium Check-In Spinner',
    '7330257731682': 'VersaPack+ Carry-On / Medium / Large',
    '7343161245794': 'Platinum Elite Carry-On / Medium Check-In Spinner Set',
    '7209952870498': 'Travelpro Altitude Organization Kit',
    '7343580086370': 'Platinum Elite Carry-On / Medium / Large Set',
    '7330299674722': 'Crew Classic Carry-On / Large Check-In Set',
    '7343584673890': 'Platinum Elite Carry-On / Medium / Large Set',
    '7321792872546': 'Platinum Elite Carry-On / Business Backpack Camouflage Set',
    '7215285272674': 'Travelpro Altitude Cord Pouch',
    '7332750721122': 'Maxlite Air V2 Carry-On / Large Check-In Set',
    '7332751868002': 'Maxlite Air V2 Carry-On / Large Check-In Set',
    '7325141631074': 'Platinum Elite Carry-On / Large Check-In Hardside Set',
    '7325347086434': 'Travelpro x SURGEON Carry-On Hardside Spinner',
    '7332753014882': 'Maxlite Air V2 Carry-On / Medium Check-In Set',
    '7332753965154': 'Maxlite Air V2 Carry-On / Medium / Large Set',
    '7231782846562': 'Platinum Elite Slim Backpack - Camouflage',
    '7209925410914': 'Travelpro Altitude Slim Expandable Laptop Backpack 20-24L',
    '7334523338850': 'Versapack+ Medium Check-in Spinner / Travelpro Altitude Large Backpack Set',
    '7335049986146': 'Platinum Elite Carry-On / UnderSeat Tote Set',
    '7327961448546': 'Platinum Elite Carry-On / Large Check-In Hardside Set',
    '7327961677922': 'Platinum Elite Carry-On / Large Check-In Hardside Set',
    '7339135696994': 'Crew Classic Compact Carry-On / Medium Check-In Set',
    '7339136876642': 'Crew Classic Compact Carry-On / Medium Check-In Set',
    '7209951166562': 'Travelpro Altitude Double Expansion Duffel',
    '7307495014498': 'Maxlite Air V2 Carry-On Hardside Spinner',
    '7342339227746': 'Platinum Elite Carry-On / Medium Check-In Hardside Set',
    '7342368817250': 'Platinum Elite Carry-On / Medium Check-In Hardside Set',
    '7330245410914': 'VersaPack+ Carry-On / Medium / Large Set',
    '7330251079778': 'VersaPack+ Carry-On / Medium / Large Set',
    '7343159443554': 'Platinum Elite Carry-On / Medium Check-In Spinner Set',
    '7343573172322': 'Platinum Elite Carry-On / Medium / Large Set',
    '7321546752098': 'Platinum Elite Carry-On / Slim Backpack Camouflage Set',
    '7202993733730': 'Roundtrip Rolling UnderSeat Carry on',
    '7345621205090': 'Platinum Elite Medium / Large Check-In Set',
    '7330323562594': 'Platinum Elite Carry-On / Business Backpack Set',
    '7331689562210': 'Travelpro x SURGEON Custom Travel Set',
    '7332749279330': 'Maxlite Air V2 Carry-On / Large Check-In Set',
    '7231655247970': 'Platinum Elite UnderSeat Tote - Camouflage',
    '7332752523362': 'Maxlite Air V2 Carry-On / Medium Check-In Set',
    '7332752916578': 'Maxlite Air V2 Carry-On / Medium Check-In Set',
    '7332755832930': 'Maxlite Air V2 Carry-On / Medium / Large Set',
    '7332760060002': 'Maxlite Air V2 Carry-On / Medium / Large Set',
    '7327959941218': 'Platinum Elite Carry-On / Large Check-In Hardside Set',
    '7270779289698': 'Platinum Elite Medium Check-In Spinner - Camouflage',
    '7270779388002': 'Platinum Elite Large Check-In Spinner - Camouflage',
    '7327961907298': 'Platinum Elite Carry-On / Large Check-In Hardside Set',
    '7339159060578': 'Crew Classic Compact Carry-On / Large Check-In Set',
    '7339986485346': 'Platinum Elite Carry-On / Medium Check-In Hardside Set',
    '7343138865250': 'Platinum Elite Carry-On / Medium Check-In Spinner Set',
    '7343152529506': 'Platinum Elite Carry-On / Medium Check-In Spinner Set',
    '7307963891810': 'Maxlite Air V2 International Carry-On Hardside Spinner',
    '7330297872482': 'Crew Classic Carry-On / Large Check-In Set',
    '7330300264546': 'Crew Classic Carry-On / Large Check-In Set',
    '7330308685922': 'Platinum Elite Carry-On / Business Backpack Set',
    '7231595708514': 'Platinum Elite Drop-Bottom Weekender - Camouflage',
    '7323923775586': 'Platinum Elite Carry-On / Medium / Large Camouflage Set',
    '7332752588898': 'Maxlite Air V2 Carry-On / Medium Check-In Set',
    '7327359860834': 'Travelpro Altitude Crossbody 6L',
    '7327895158882': 'Travelpro Altitude All-Purpose Full Expansion Laptop Backpack 24-34L',
    '7335061749858': 'Platinum Elite Carry-On / UnderSeat Tote Set',
    '7338753523810': 'Maxlite  Carry-On Spinner / Laptop Backpack Set',
    '7328188432482': 'Travelpro Altitude Day-to-Day Duffel',
    '7329628422242': 'Crew Classic Carry-On / Medium Check-In Set',
    '7343150923874': 'Platinum Elite Carry-On / Medium Check-In Spinner Set',
    '7330262450274': 'VersaPack+ Carry-On / Medium / Large',
    '7343583232098': 'Platinum Elite Carry-On / Medium / Large Set',
    '7174756860002': 'Travelpro Essentials SparePack Foldable Duffel 2.0',
    '2106933870690': 'Maxlite 5 Medium Check-In Spinner',
    '7151421227106': 'VersaPack+ Compact Carry-On Spinner',
    '2106966605922': 'Maxlite 5 International Carry-On Spinner',
    '7343157477474': 'Platinum Elite Carry-On / Medium Check-In Spinner Set',
    '2107040399458': 'Maxlite 5 Carry-On Rolling Underseat Bag',
    '7343586213986': 'Platinum Elite Carry-On / Medium / Large Set',
    '2107183792226': 'Maxlite 5 Carry-On Rolling Garment Bag',
    '7323800272994': 'Platinum Elite Carry-On / Medium Check-In Camouflage Set',
    '2107450064994': 'Bold by Travelpro Carry-On Spinner',
    '7202488778850': 'Platinum Elite UnderSeat Tote',
    '2107479490658': 'Bold By Travelpro Large Check-In Rollaboard',
    '2104883150946': 'Platinum Elite Carry-On Business Plus Spinner',
    '2211939909730': 'Platinum Elite International Carry-On Rollaboard',
    '7327834243170': 'Travelpro Altitude Convertible Duffel/Backpack',
    '2227516833890': 'Travelpro Essentials Large Expandable/Compressible Packing Cube',
    '2105060294754': 'Platinum Elite Large Check-In Spinner',
    '3878339477602': 'Travelpro Essentials Shoe Bags 2 Pack',
    '7202495922274': 'Platinum Elite Drop-Bottom Weekender',
    '4322499264610': 'Travelpro Essentials Leather Luggage Tag',
    '2105263423586': 'Platinum Elite Carry-On Rolling Garment Bag',
    '4330976837730': 'Travelpro Essentials MaxAccess Cubes Large Organizer',
    '7202295742562': 'Platinum Elite Slim Backpack',
    '4330988699746': 'Travelpro Essentials MaxAccess Cubes Large Shoe Organizer',
    '2106858143842': 'Maxlite 5 Breakaway 21" / 25" Set',
    '4436406992994': 'Travelpro Essentials XL Expandable/Compressible Packing Cube',
    '7329627701346': 'Crew Classic Carry-On / Medium Check-In Set',
    '4482401992802': 'Travelpro x Travel + Leisure Convertible Backpack',
    '4482406252642': 'Travelpro x Travel + Leisure UnderSeat Tote',
    '4482410578018': 'Travelpro x Travel + Leisure Slim Backpack',
    '4482412347490': 'Travelpro x Travel + Leisure Drop-Bottom Weekender',
    '4482417852514': 'Travelpro x Travel + Leisure Compact Carry-On Spinner',
    '4482419130466': 'Travelpro x Travel + Leisure Carry-On Spinner',
    '4482421129314': 'Travelpro x Travel + Leisure Medium Check-In Spinner',
    '4482423128162': 'Travelpro x Travel + Leisure Large Check-In Trunk Spinner',
    '4485486706786': 'Bold 22" / 28" Rollaboard Set',
    '4488975712354': 'Travelpro x Travel + Leisure Carry-On / Medium Check-In Set',
    '4488988459106': 'Travelpro x Travel + Leisure Carry-On / Large Check-In Trunk Set',
    '4488989769826': 'Travelpro x Travel + Leisure Compact Carry-On / Medium Check-In Set',
    '4488992850018': 'Travelpro x Travel + Leisure Compact Carry-On / Large Check-In Trunk Set',
    '4513677213794': 'Roundtrip Carry-On / Medium Check-In Hardside Set',
    '4527217672290': 'Travelpro x Travel + Leisure Medium Check-In Expandable Spinner',
    '4527217868898': 'Travelpro x Travel + Leisure Large Check-In Trunk Spinner',
    '4530815271010': 'Platinum Elite Carry-On Hardside Spinner',
    '4535556669538': 'Platinum Elite Compact Carry-On Hardside Spinner',
    '4535556964450': 'Platinum Elite Medium Check-In Hardside Spinner',
    '4537178390626': 'Platinum Elite Large Check-In Hardside Spinner',
    '4538767147106': 'Platinum Elite Compact Carry-On Business Plus Hardside Spinner',
    '4538772160610': 'Platinum Elite Carry-On Business Plus Hardside Spinner',
    '6546897338466': 'Travelpro Essentials 2-in-1 Travel Tote & Cooler',
    '6548028457058': 'Crew Executive Choice 3 Slim Laptop Backpack',
    '6548063584354': 'Crew Executive Choice 3 Women’s Tote',
    '6548254785634': 'Crew Executive Choice 3 Medium Top Load Backpack',
    '6548632240226': 'Crew Executive Choice 3 Large Travel Backpack',
    '6578390958178': 'Roadtrip 30" Drop-Bottom Rolling Duffel with Packing Cubes',
    '6578410324066': '3 Pack Roadtrip Large Packing Cubes',
    '6588696461410': 'Travelpro x Travel + Leisure Slim Backpack',
    '6588697018466': 'Travelpro x Travel + Leisure Medium Check-In Expandable Spinner',
    '6588697444450': 'Travelpro x Travel + Leisure Large Check-In Trunk Spinner',
    '6588698853474': 'Travelpro x Travel + Leisure Compact Carry-On Expandable Spinner',
    '6588699181154': 'Travelpro x Travel + Leisure Carry-On Expandable Spinner',
    '6588710944866': 'Travelpro x Travel + Leisure Carry-on/ Large Check-in Trunk Spinner - Luggage Set',
    '6588711370850': 'Travelpro x Travel + Leisure Compact Carry-on/ Large Check-in Trunk Spinner - Luggage Set',
    '6602747576418': 'Platinum Elite Compact Carry-On / Large Check-In Hardside Set',
    '6734366015586': 'Maxlite 5 Drop-Bottom Weekender',
    '6790148980834': "Maxlite Women's Tote",
    '7307855921250': 'Maxlite Air V2 Medium Check-In Hardside Spinner',
    '6808100929634': 'Travelpro x Travel + Leisure Carry-on/ Large Check-in Trunk Spinner - Luggage Set',
    '6808101879906': 'Travelpro x Travel + Leisure Compact Carry-On Expandable Spinner',
    '6808103256162': 'Travelpro x Travel + Leisure Compact Carry-on/ Large Check-in Trunk Spinner - Luggage Set',
    '6808104992866': 'Travelpro x Travel + Leisure Large Check-In Trunk Spinner',
    '6808105091170': 'Travelpro x Travel + Leisure Medium Check-In Expandable Spinner',
    '6808105418850': 'Travelpro x Travel + Leisure Slim Backpack',
    '6809338970210': 'Maxlite Laptop Backpack',
    '6812850585698': 'Maxlite Checked Rolling Garment Bag',
    '6878384259170': 'Travelpro x Travel + Leisure Carry-On / UnderSeat Tote Set',
    '6878439211106': 'Travelpro x Travel + Leisure Compact Carry-On / UnderSeat Tote Set',
    '6878441767010': 'Travelpro x Travel + Leisure Large Check-In / Drop-Bottom Weekender Set',
    '6878443274338': 'Travelpro x Travel + Leisure Medium Check-in Spinner and Drop-Bottom Weekender Bag Luggage Set',
    '6958642692194': 'Travelpro Essentials Drop-in Garment Sleeve',
    '6968181194850': 'Crew Classic UnderSeat Tote',
    '6968381309026': 'Crew Classic Carry-On Rollaboard',
    '6968391401570': 'Crew Classic Carry-On Spinner',
    '6968392155234': 'Crew Classic Compact Carry-On Spinner',
    '6968413618274': 'Crew Classic Medium Check-In Spinner',
    '6968413978722': 'Crew Classic Large Check-In Spinner',
    '6968715477090': 'Crew Classic Rolling UnderSeat Carry-On',
    '6998502670434': 'Crew Classic UnderSeat Tote / Carry-On Set',
    '2106952351842': 'Maxlite 5 Medium Check-In Rollaboard',
    '7151421620322': 'VersaPack+ Large Check-In Spinner',
    '7151421685858': 'VersaPack+ UnderSeat Tote',
    '7171362881634': 'VersaPack+ Carry-On / Medium Check-In Set',
    '7171506896994': 'VersaPack+ Compact Carry-On / Medium Check-In Set',
    '7171515187298': 'VersaPack+ Compact Carry-On / Large Check-In Set',
    '7171528097890': 'VersaPack+ Carry-On / Large Check-In Set',
    '7174740312162': 'Travelpro Essentials SparePack Foldable Backpack 2.0',
    '7174751977570': 'Travelpro Essentials SparePack Foldable Tote 2.0',
    '7209951920226': 'Travelpro Altitude Full Expansion Brief',
    '2107039416418': 'Maxlite 5 Carry-On Rolling Tote',
    '2107160625250': 'Maxlite 5 Soft Tote',
    '7331393437794': 'Platinum Elite Carry-On / Large Check-In Camouflage Set',
    '7332752261218': 'Maxlite Air V2 Carry-On / Large Check-In Set',
    '2107471855714': 'Bold by Travelpro Medium Check-In Spinner',
    '2107480408162': 'Bold by Travelpro Large Check-In Drop-Bottom Rolling Duffel',
    '2104948326498': 'Platinum Elite Medium Check-In Spinner',
    '2104952422498': 'Platinum Elite Expandable Business Brief',
    '3875262005346': 'Maxlite 5 Floating On Air Set',
    '3878347309154': 'Travelpro Essentials Split Case Toiletry Bag',
    '7335766294626': 'Platinum Elite Carry-On / Slim Backpack Set',
    '7335819247714': 'Platinum Elite Carry-On / Slim Backpack Set',
    '4330983227490': 'Travelpro Essentials MaxAccess Cubes Toiletry Organizer',
    '4373934047330': 'Travelpro Essentials Waist Bag',
    '2106881704034': 'Maxlite 5 Carry-On Rollaboard',
    '4462384283746': 'Maxlite 5 21" / 29" Set',
    '7151421423714': 'VersaPack+ Carry-On Spinner',
    '7151421456482': 'VersaPack+ Medium Check-In Spinner',
    '7202409119842': 'Platinum Elite Soft Duffel',
    '7343582052450': 'Platinum Elite Carry-On / Medium / Large Set',
    '2107397898338': 'Bold By Travelpro Carry-On Rollaboard',
    '2107450785890': 'Bold By Travelpro Medium Check-In Rollaboard',
    '2107529494626': 'Platinum Elite Trend Setter Set',
    '2227502481506': 'Travelpro Essentials Medium Expandable/Compressible Packing Cube',
    '7334511804514': 'Crew Classic Carry-on Spinner / Travelpro Altitude Large Backpack Set',
    '2105128190050': 'Platinum Elite International Carry-On Spinner',
    '2105359433826': 'Platinum Elite 50” Check-In Rolling Garment Bag',
    '7339157684322': 'Crew Classic Compact Carry-On / Large Check-In Set',
    '4436413186146': 'Travelpro Essentials 3 Pack Expandable/Compressible  Packing Cube Set (M/L/XL)',
    '6808100601954': 'Travelpro x Travel + Leisure Carry-On Expandable Spinner',
    '2106967949410': 'Maxlite 5 International Carry-On Rollaboard',
    '7345813225570': 'Platinum Elite Medium / Large Check-In Set',
    '2104797200482': 'Platinum Elite Carry-On Rollaboard',
    '2225953669218': 'Travelpro Essentials 13"/14" Laptop Sleeve',
    '2105110364258': 'Platinum Elite Carry-On Spinner Tote',
    '4322824388706': 'Travelpro Essentials MaxAccess Cubes Small Organizer',
    '4373937815650': 'Travelpro Essentials Garment Packing Folder',
    '6802391072866': 'Maxlite 5 Compact Carry-On Spinner',
    '7343579365474': 'Platinum Elite Carry-On / Medium / Large Set',
    '2104408211554': 'Travelpro Platinum Elite Carry-On Spinner',
    '7332758519906': 'Maxlite Air V2 Carry-On / Medium / Large Set',
    '4322505097314': 'Travelpro Essentials Leather Passport Cover',
    '7339961057378': 'Platinum Elite Carry-On / Medium Check-In Hardside Set',
    '2107399045218': 'Bold By Travelpro Computer Backpack With Compartments',
    '2227517751394': 'Travelpro Essentials 3 Pack Packing Cube Set (S/M/L)',
    '2106832552034': 'Maxlite 5 Carry-On Spinner',
    '7205198463074': 'Travelpro Altitude Medium Expandable Laptop Backpack 25-30L',
    '4330979131490': 'Travelpro Essentials MaxAccess Cubes Deluxe Hanging Toiletry Organizer',
    '7231753912418': 'Platinum Elite Business Backpack - Camouflage',
    '2107179008098': 'Maxlite 5 Large Check-In Spinner',
    '3879519223906': 'Travelpro Essentials Washable Travel Laundry Bag',
    '4454501285986': 'Platinum Elite Backpack 1.0 / 21" Carry-On Set',
}

# if __name__ == "__main__":
#   # Set the file name to save the reviews under the current date
#   today = date.today()
#   fName = f"TravelPro-US-Reviews-{today}.txt"
#   file = open(fName, "w")
#   file.write("------------------------------------------------------------------\n")
#   file.write("Travelpro US store reviews (travelpro.com)\n")
#   file.write("------------------------------------------------------------------\n\n")

#   # Loop through each product ID in the PRODUCTS dictionary
#   # and fetch the reviews for each product.
#   for id in PRODUCTS:
#     reviews = fetch_all_reviews(APP_KEY, id)
#     file.write(f"PRODUCT NAME: {PRODUCTS[id]}\n")
#     file.write(f"Total reviews: {len(reviews)}\n\n")
#     for i in range(len(reviews)):
#       print_review(i, reviews, file, PRODUCTS[id])

#   # Remember to close the file after writing to it.
#   file.close()

# if __name__ == "__main__":
#     from datetime import date

#     today = date.today()
#     fName = f"TravelPro-US-Reviews-{today}.txt"

#     with open(fName, "w") as file:
#         file.write("------------------------------------------------------------------\n")
#         file.write("Travelpro US store reviews (travelpro.com)\n")
#         file.write("------------------------------------------------------------------\n\n")

#         # reviews = fetch_all_reviews(APP_KEY, API_SECRET)  # no product filter
#         reviews = discover_product_ids_with_reviews(APP_KEY)
#         # group reviews by product_id
#         by_product = {}
#         for rev in reviews:
#             pid = rev.get("product_id")
#             pname = rev.get("product_title", f"Product {pid}")
#             by_product.setdefault((pid, pname), []).append(rev)

#         for (pid, pname), items in by_product.items():
#             file.write(f"PRODUCT NAME: {pname} (ID: {pid})\n")
#             file.write(f"Total reviews: {len(items)}\n\n")
#             for i, r in enumerate(items):
#                 print_review(i, items, file, pname)
#             file.write("\n\n")
#     file.close()

if __name__ == "__main__":
    out_path = f"TravelPro-Product-Displayed-Reviews-{date.today()}.txt"

    with open(out_path, "w", encoding="utf-8") as file:
        file.write("------------------------------------------------------------------\n")
        file.write("Travelpro US product reviews (PUBLISHED / DISPLAYED on PDPs)\n")
        file.write("------------------------------------------------------------------\n\n")

        try:
            product_ids = discover_product_ids_with_reviews(APP_KEY, per_page=PER_PAGE_BOTTOMLINES)
        except Exception as e:
            file.write(f"[FATAL] Failed to discover product ids: {e}\n")
            raise

        if not product_ids:
            file.write("[INFO] No product IDs with published reviews were discovered.\n")
        else:
            file.write(f"[INFO] Discovered {len(product_ids)} product IDs with reviews.\n\n")

        for pid in product_ids:
            try:
                reviews, product_name = fetch_displayed_product_reviews(
                    APP_KEY, pid, per_page=PER_PAGE_WIDGET, sort="date", direction="desc"
                )
            except Exception as e:
                file.write(f"[WARN] Failed to fetch reviews for product_id={pid}: {e}\n\n")
                continue

            if not reviews:
                file.write(f"[INFO] No published reviews returned for product_id={pid}; skipping.\n")
                continue

            pName = product_name or f"Product {pid}"
            file.write(f"PRODUCT NAME: {pName} (ID: {pid})\n")
            file.write(f"Total reviews: {len(reviews)}\n\n")

            for r in reviews:
                print_review(r, file, pName)

            file.write("\n")

    print(f"Done. Wrote: {out_path}")

Done. Wrote: TravelPro-Product-Displayed-Reviews-2025-10-01.txt


In [ ]:
# TravelPro CA STORE

# App Key for Yotpo API
# This key is used to authenticate requests to the Yotpo API.
APP_KEY = "3XUPi9vAFJphnYsXrhusGWZahu5xJCUHatWERl89"

# This dictionary contains product IDs and their corresponding names.
PRODUCTS = {
    '7640326438975': 'Travelpro Altitude Organization Kit',
    '7640326406207': 'Travelpro Altitude Full Expansion Brief',
    '7640326537279': 'Travelpro Altitude Slim Expandable Laptop Backpack 20-24L',
    '7640326471743': 'Travelpro Altitude Large Expandable Laptop Backpack 30-36L',
    '7640326504511': 'Travelpro Altitude Medium Expandable Laptop Backpack 25-30L',
    '7658854907967': 'Platinum Elite Medium Check-In Spinner - Camouflage',
    '7658855006271': 'Platinum Elite UnderSeat Tote - Camouflage',
    '7658855104575': 'Platinum Elite Large Check-In Spinner - Camouflage',
    '7658855137343': 'Travelpro Platinum Elite Carry-On Spinner - Camouflage',
    '7732641431615': 'Platinum Elite Camouflage 2-Piece Bundle | Backpack & Carry-On',
    '7752912339007': 'Maxlite Air V2 International Carry-On Hardside Spinner',
    '7752912371775': 'Maxlite Air V2 Large Check-In Hardside Spinner',
    '7752912404543': 'Maxlite Air V2 Medium Check-In Hardside Spinner',
    '7752912633919': 'Maxlite Air V2 Compact Carry-On Hardside Spinner',
    '7760498655295': 'Maxlite Air V2 Compact Carry-On / Large Check-In Set',
    '7760499081279': 'Maxlite Air V2 Compact Carry-On / Large Check-In Set',
    '7760499343423': 'Maxlite Air V2 Compact Carry-On / Large Check-In Set',
    '7760500097087': 'Maxlite Air V2 Compact Carry-On / Large Check-In Set',
    '7760500228159': 'Maxlite Air V2 Compact Carry-On / Large Check-In Set',
    '7760500326463': 'Maxlite Air V2 Compact Carry-On / Large Check-In Set',
    '7760500523071': 'Maxlite Air V2 Compact Carry-On / Medium Check-In Set',
    '7760500588607': 'Maxlite Air V2 Compact Carry-On / Medium Check-In Set',
    '7760500654143': 'Maxlite Air V2 Compact Carry-On / Medium Check-In Set',
    '7760500785215': 'Maxlite Air V2 Compact Carry-On / Medium Check-In Set',
    '7760501407807': 'Maxlite Air V2 Compact Carry-On / Medium Check-In Set',
    '7760501571647': 'Maxlite Air V2 Compact Carry-On / Medium Check-In Set',
    '7760501669951': 'Maxlite Air V2 Compact Carry-On / Medium / Large Set',
    '7760501932095': 'Maxlite Air V2 Compact Carry-On / Medium / Large Set',
    '7760502161471': 'Maxlite Air V2 Compact Carry-On / Medium / Large Set',
    '7760502325311': 'Maxlite Air V2 Compact Carry-On / Medium / Large Set',
    '7760502489151': 'Maxlite Air V2 Compact Carry-On / Medium / Large Set',
    '7760502915135': 'Maxlite Air V2 Compact Carry-On / Medium / Large Set',
    '7648622870591': 'Travelpro Essentials 2-in-1 Travel Tote & Cooler',
    '7658854940735': 'Platinum Elite Business Backpack - Camouflage',
    '7658854973503': 'Platinum Elite Soft Duffel - Camouflage',
    '1683435880511': 'Platinum Elite 22” Expandable Carry-On Rollaboard',
    '1683796099135': 'Platinum Elite 21” Expandable Carry-On Spinner',
    '1684051918911': 'Platinum Elite 25” Expandable Spinner',
    '1685213511743': 'Platinum Elite 29” Expandable Spinner',
    '1685230485567': 'Platinum Elite Regional Carry-On Duffle',
    '1686056108095': 'Platinum Elite Carry-On Rolling Garment Bag',
    '1686057582655': 'Platinum Elite 50” Rolling Garment Bag',
    '1686089105471': 'Maxlite 5 Soft Tote',
    '1686129868863': 'Maxlite 5 19" International Carry-On Expandable Spinner',
    '1686146580543': 'Maxlite 5 25" Expandable Spinner',
    '1686155853887': 'Maxlite 5 29" Expandable Spinner',
    '1686360621119': 'Maxlite 5 Rolling Underseat Bag',
    '1686473146431': 'Maxlite 5 26" Expandable Rollaboard',
    '1686477733951': 'Maxlite 5 20" International Carry-On Expandable Rollaboard',
    '1687563075647': 'Maxlite 5 Breakaway - Luggage Set',
    '1817420726335': "Platinum Elite Women's Brief",
    '1817450020927': "Platinum Elite Women's Crossbody",
    '1878343155775': 'Travelpro Essentials Medium Expandable/Compressible Packing Cube',
    '1878357311551': 'Travelpro Essentials Large Expandable/Compressible Packing Cube',
    '1878392471615': 'Travelpro Essentials 3 Pack Packing Cube Set (small/med/large)',
    '1976967168063': 'Platinum Elite Expandable Business Brief',
    '1977011568703': 'Platinum Elite Business Backpack',
    '4162147778623': 'Travelpro Essentials Shoe Bags 2 Pack',
    '4162287075391': 'Travelpro Essentials Split Case Toiletry Bag',
    '4162292285503': 'Travelpro Essentials Washable Laundry Bag',
    '4162296184895': 'Travelpro Essentials Security Waist Pouch',
    '4162300379199': 'Travelpro Essentials Security Neck Pouch',
    '4185244696639': 'Maxlite5 Carry Me Away - Luggage Set',
    '4185290768447': 'Maxlite 5 Floating On Air - Luggage Set',
    '4185434423359': 'Platinum Elite: First Class - Luggage Set',
    '4309797142591': 'Travelpro Essentials Leather Luggage Tag',
    '4311273996351': 'Travelpro Essentials Leather Passport Cover',
    '4311294804031': 'Travelpro Essentials Personalization Kit',
    '4311396843583': 'Travelpro Essentials MaxAccess Cubes Small Organizer',
    '4311416864831': 'Travelpro Essentials MaxAccess Cubes Large Organizer',
    '4311445602367': 'Travelpro Essentials MaxAccess Cubes Toiletry Organizer',
    '4311466246207': 'Travelpro Essentials MaxAccess Cubes Deluxe Hanging Toiletry Organizer',
    '4311509696575': 'Travelpro Essentials MaxAccess Cubes Large Shoe Organizer',
    '4750612791359': 'Platinum Elite Trend Setter - Luggage Set',
    '4776667283519': 'Platinum Elite 21/25/29 - Luggage Set',
    '4788995883071': 'Travelpro x Travel + Leisure Convertible Backpack',
    '4788996177983': 'Travelpro x Travel + Leisure UnderSeat Tote',
    '4788996309055': 'Travelpro x Travel + Leisure Slim Backpack',
    '4788996833343': 'Travelpro x Travel + Leisure Drop-Bottom Weekender',
    '4788997521471': "Travelpro x Travel + Leisure Women's Convertible Tote",
    '4788997816383': 'Travelpro x Travel + Leisure Compact Carry-On Expandable Spinner',
    '4788997947455': 'Travelpro x Travel + Leisure Carry-On Expandable Spinner',
    '4788998111295': 'Travelpro x Travel + Leisure Medium Check-In Expandable Spinner',
    '4788998766655': 'Travelpro x Travel + Leisure Large Check-In Trunk Spinner',
    '4788999094335': 'Travelpro x Travel + Leisure Carry-On/Medium Check-In Spinner - Luggage Set',
    '4788999356479': 'Travelpro x Travel + Leisure Carry-on/Large Check-in Trunk Spinner - Luggage Set',
    '4789000077375': 'Travelpro x Travel + Leisure Compact Carry-on/Checked Medium Spinner - Luggage Set',
    '4789000601663': 'Travelpro x Travel + Leisure Compact Carry-on/Large Check-in Trunk Spinner - Luggage Set',
    '4947561185343': 'Travelpro Essentials Waist Bag',
    '4947561218111': 'Travelpro Essentials 3 Pack Expandable/Compressible  Packing Cube Set (M/L/XL)',
    '4947561250879': 'Travelpro Essentials XL Expandable/Compressible Packing Cube',
    '4947561807935': 'Travelpro Essentials Garment Packing Folder',
    '6627045900351': 'Platinum Elite Large Check-In Hardside Spinner',
    '6627046260799': 'Platinum Elite Carry-On Hardside Spinner',
    '6627046621247': 'Platinum Elite Medium Check-In Hardside Spinner',
    '6627047440447': 'Platinum Elite Compact Carry-On Hardside Spinner',
    '6627047604287': 'Platinum Elite Carry-On Business Plus Hardside Spinner',
    '6627048456255': 'Platinum Elite Carry-On / Medium Check-in Hardside Luggage Set',
    '6627048718399': 'Platinum Elite Compact Carry-On / Medium Check-in Hardside Luggage Set',
    '7022207991871': 'Travelpro x Travel + Leisure Carry-on Spinner and UnderSeat Tote Luggage Set',
    '7022887567423': 'Travelpro x Travel + Leisure Compact Carry-on Spinner and UnderSeat Tote Luggage Set',
    '7022913978431': 'Travelpro x Travel + Leisure Medium Check-in Spinner and Drop-Bottom Weekender Bag Luggage Set',
    '7022930788415': 'Travelpro x Travel + Leisure Large Check-in Spinner and Drop-Bottom Weekender Bag Luggage Set',
    '7023045214271': 'Maxlite 5 Drop-Bottom Weekender',
    '7104462520383': 'Platinum Elite Carry-On / Large Check-In Hardside Set',
    '7104462553151': 'Platinum Elite Compact Carry-On / Large Check-In Hardside Set',
    '7109878120511': "Maxlite Women's Tote",
    '7109878153279': 'Maxlite 5 Compact Carry-On Expandable Spinner',
    '7109878186047': 'Maxlite Laptop Backpack',
    '7113207218239': 'Crew Executive Choice 3 Women’s Tote',
    '7113207251007': 'Crew Executive Choice 3 Large Backpack',
    '7113207283775': 'Crew Executive Choice 3 Slim Backpack',
    '7113207316543': 'Crew Executive Choice 3 Medium Top Load Backpack',
    '7116727582783': 'Platinum Elite Compact Carry-On Business Plus Hardside Spinner',
    '1683574652991': 'Platinum Elite International Expandable Carry-On Spinner',
    '1683487621183': 'Platinum Elite Carry-On Spinner Tote',
    '7144729411647': 'Maxlite 5 Carry-On Rolling Garment Bag',
    '7248487546943': 'Connoisseur 4 31.5 Inch Large Expandable Spinner Suitcase',
    '7248520282175': 'Connoisseur 4 - 27 Inch Medium Expandable Spinner Suitcase',
    '7248523460671': 'Connoisseur 4 - 21.75 Inch Carry-On Expandable Spinner Suitcase',
    '7248529293375': 'Pathways 3.0  - 29" Large Check-in Hardside Expandable Spinner',
    '7248529358911': 'Pathways 3.0 - 25" Medium Check-in Hardside Expandable Spinner',
    '7298691891263': 'Crew Classic UnderSeat Tote',
    '7298691924031': 'Crew Classic Rolling UnderSeat Carry-on',
    '7298691956799': 'Crew Classic Large Check-in Expandable Spinner',
    '7298691989567': 'Crew Classic Carry-On Expandable Spinner',
    '7298692350015': 'Crew Classic Carry-On Expandable Rollaboard',
    '7298692382783': 'Crew Classic Medium Check-in Expandable Spinner',
    '7298692513855': 'Crew Classic Compact Carry-On Expandable Spinner',
    '7369452388415': 'Crew Classic Carry-On / Medium Check-in Luggage Set',
    '7369452421183': 'Crew Classic Compact Carry-On / Large Check-in Luggage Set',
    '7369452453951': 'Crew Classic UnderSeat Tote / Carry-On Spinner Luggage Set',
    '7369452486719': 'Crew Classic Carry-On / Large Check-in Luggage Set',
    '7369453207615': 'Crew Classic Compact Carry-On / Medium Check-in Luggage Set',
    '7472490315839': 'RTS6902009',
    '7570063261759': 'VersaPack+ Medium Check-In Spinner',
    '7570063294527': 'VersaPack+ Large Check-In Spinner',
    '7570063327295': 'VersaPack+ Carry-On Spinner',
    '7570063360063': 'VersaPack+ Compact Carry-On Spinner',
    '7570064474175': 'VersaPack+ UnderSeat Tote',
    '7570064769087': 'VersaPack+ Carry-On / Large Check-In Spinner Set',
    '7570064867391': 'VersaPack+ Compact Carry-On / Large Check-In Spinner Set',
    '7570064900159': 'VersaPack+ Carry-On / Medium / Large Check-In Spinner Set',
    '7570065227839': 'VersaPack+ Carry-On / Medium Check-In Spinner Set',
    '7570066374719': 'VersaPack+ Compact Carry-On / Medium Check-In Spinner Set',
    '7578782236735': 'Platinum Elite UnderSeat Tote',
    '7578782269503': 'Platinum Elite Business Backpack',
    '7578782302271': 'Platinum Elite Drop-Bottom Weekender',
    '7578782335039': 'Platinum Elite Soft Duffel',
    '7578782400575': 'Platinum Elite Slim Backpack',
    '7579804434495': 'Travelpro Essentials SparePack Foldable Duffel 2.0',
    '7579804467263': 'Travelpro Essentials SparePack Foldable Backpack 2.0',
    '7579804500031': 'Travelpro Essentials SparePack Foldable Tote 2.0',
    '7640263721023': 'Platinum Elite Drop-Bottom Weekender - Camouflage',
    '7640263753791': 'Platinum Elite Slim Backpack - Camouflage',
    '7640326340671': 'Travelpro Altitude Cord Pouch',
    '7640326373439': 'Travelpro Altitude Double Expansion Duffel',
    '7144729444415': 'Maxlite 5 Carry-On Rolling Tote',
    '1683583369279': 'Platinum Elite 20” Expandable Business Plus Carry-On Spinner',
    '7121829363775': 'Maxlite Checked Rolling Garment Bag',
}

if __name__ == "__main__":
  # Set the file name to save the reviews under the current date
  today = date.today()
  fName = f"TravelPro-CA-Reviews-{today}.txt"
  file = open(fName, "w")
  file.write("------------------------------------------------------------------\n")
  file.write("Travelpro CA store reviews (travelpro.ca)\n")
  file.write("------------------------------------------------------------------\n\n")

  # Loop through each product ID in the PRODUCTS dictionary
  # and fetch the reviews for each product.
  for key in PRODUCTS:
    reviews = fetch_all_reviews(APP_KEY, key)
    file.write(f"PRODUCT NAME: {PRODUCTS[key]}\n")
    file.write(f"Total reviews: {len(reviews)}\n\n")
    for i in range(len(reviews)):
      print_review(i, reviews, file, PRODUCTS[key])

  # Remember to close the file after writing to it.
  file.close()

In [ ]:
# TravelPro EU STORE

# App Key for Yotpo API
# This key is used to authenticate requests to the Yotpo API.
APP_KEY = "U6eCTWq9q9EAD6F9PWWheSf6swvAb5NFiMNukpeN"

# This dictionary contains product IDs and their corresponding names.
PRODUCTS = {
    '14977855291717': 'ML5#5 ZIPPER SLDER LOGO BLK',
    '14977855226181': 'ML5 LFT/REAR RG/FRONT SPINNER',
    '14977855455557': 'ML5#8 LCKNG SLDR LOGO BLK',
    '14977855324485': 'ML5#5 ZIPPER SLDER LOGO SL GRN',
    '14977855390021': 'ML5#8 ZIPPER SLDER LOGO BLK',
    '14977855652165': 'FC5 REMOVABLE SHOULDER STRAP',
    '14977855979845': 'FC5 RGT FRNT/BACK LFT SPNR WHL',
    '14977856012613': '# 5 SLIDER WITH TRAVELPRO LOGO',
    '14977856143685': 'FC5# 8 HS LCK SLDR W LOGO PULL',
    '14977856209221': 'FC5 #8 EXPANSION ZIPPER RUBBER',
    '14977856274757': 'FC5#10  SLDR W LOGO PULL',
    '14977856340293': 'FC5#10 LOCKNG SLDR W LOGO PULL',
    '14977856373061': 'SIDE CARRY HANDLE',
    '14977856471365': 'TL Side Foot',
    '14977856602437': 'PE BOTTOM TRAY WITH (4) WHEELS',
    '14977856700741': 'PE REMOVABLE SHOULDER STRAP BK',
    '14977856799045': 'PE LEFT BOTTOM FOOT',
    '14977856930117': 'PE RIGHT BOTTOM FOOT',
    '14977856995653': 'PE RET HDL 20IN INTL SPNR',
    '14977857061189': 'PE 3-STG RET HNDL SPNR-25IN',
    '14977857192261': 'PE 3-STG RET HNDL SPNR-29IN',
    '14977857225029': 'PLAT ELITE TOP CARRY HNDL BLK',
    '14977857323333': 'PLAT ELITE TOP CARRY HNDL ESPR',
    '14977857421637': 'PLAT ELITE TOP CARRY HNDL BORD',
    '14977857487173': 'PLAT ELITE SIDE CARRY HNDL BLK',
    '14977857618245': 'PLAT ELITE SIDE CARRY HNDL ESP',
    '14977857651013': 'PLAT ELITE SIDE CARRY HNDL BRD',
    '14977857782085': 'PE 3-STG 20IN RET HANDLE',
    '14977857880389': '3 STG RETRACT HANDLE SYS 50"',
    '14977857913157': 'PE 21 INT SPIN TRAY',
    '14977858011461': 'PE LEFT WHEEL HOUS ASSEM/WHEEL',
    '14977858109765': 'PE RGHT WHEEL HOUS ASSEM/WHEEL',
    '14977858208069': 'PE LEFT FRONT SPNR WHEEL',
    '14977858273605': 'PE LEFT REAR SPNR WHEEL',
    '14977858339141': 'PE RIGHT FRONT SPNR WHEEL',
    '14977858470213': 'PE RIGHT REAR SPNR WHEEL',
    '14977858502981': 'PE #5 MTL SLDR /LOGO PULL BLK',
    '14977858601285': 'PE #5 MTL SLDR /LOGO PULL ESPR',
    '14977858666821': 'PE # 8 SLDR INTLCK WLOGO BLK',
    '14977858732357': 'PE #8 SLDR INTLCK W LOGO ESP',
    '14977858896197': 'PE # 8 SLDR INTLCK WLOGO BORD',
    '14977859027269': 'PE #8 SLDR W LOGO PUL BLK',
    '14977859092805': 'PE #8 SLDR W LOGO PUL ESPR',
    '14977859191109': 'PE #8 SLDR W LOGO PUL BORD',
    '14977859289413': 'PE #10 TSA SLDR LEFT TSA BLK',
    '14977859354949': 'PE #10 TSA SLDR LEFT TSA ESP',
    '14977859453253': 'PE #10 TSA SLDR LEFT TSA BORD',
    '14977859518789': 'PE #10 TSA SLDR RGT TSA BLK',
    '14977859617093': 'PE #10 TSA SLDR RGT TSA BORD',
    '14977859715397': 'ADJ ATTACH CLIP BASE BLK',
    '14977859780933': '75MM INLINE SKT WHL W/BEARINGS',
    '14977859879237': 'FC5 BOTTOM HANDLE CUP',
    '14977859944773': '#5 SLDER W/ GENERIC PULL-SLVER',
    '14977860043077': '#8 SLIDER W/LOGO PULL BLK',
    '14977860108613': 'WAB6 HS TP&SD HNDL ASM KIT BLK',
    '14977860206917': 'SHOULDER STRAP U/SEAT TOTE BLK',
    '14977860272453': 'SHOULDER STRAP U/SEAT PAT BLUE',
    '14977860337989': 'FC5 SLIM RB J-HOOK STRAP',
    '14977860436293': 'ML AIR #8 ZIP/TSA LOG ASSY BLK',
    '14977860501829': '#5 INT ZIPPER SLIDE PULL ASSM',
    '14977860600133': 'ML AIR #5 ZIP/PULL ASSY BLK',
    '14977860632901': 'TSA LOCK C/I ONLY',
    '14977860698437': 'ZIP REPAIR KIT C/O SPIN AND RB',
    '14977860763973': 'ZIP REPAIR KIT C/I SPINNERS',
    '14977860829509': '#5 INTERIOR ZIPPER PULL',
    '14977860927813': 'PE HS #8 ZIP SLD & TSA ASM BLK',
    '14977861026117': 'PE HS #8 EXT ZIP&PULL ASM BLK',
    '14977861091653': 'PE HS #3 INT ZIP&PULL ASM BLK',
    '14977861157189': 'FC5 SLIM RB ATTM CLIPBASE ASSM',
    '14990665711941': 'Handling Fee - 10',
    '15040081559877': 'Maxlite Air V2 Medium Check-In Hard Shell Spinner',
    '15040081625413': 'Maxlite Air V2 Cabin Hard Shell Spinner',
    '15040081658181': 'Maxlite Air V2 Slim Cabin Hard Shell Spinner',
    '15040082936133': 'Maxlite Air V2 Large Check-In Hard Shell Spinner',
    '3723241783348': 'Maxlite 5 Rolling Cabin Garment Bag (41 x 56 x 22 cm)',
    '3723242537012': 'Maxlite 5 Soft Tote (28 x 46 x 20 cm)',
    '3723243323444': 'Maxlite 5 Bi-Fold Hanging Garment Bag (56 x 56 x 18 cm)',
    '3723244929076': 'Maxlite 5 Rolling Cabin Tote (40 x 42 x 22 cm)',
    '3723245289524': 'Maxlite 5 Medium Expandable Soft Shell Spinner 69cm (69 x 47 x 29 cm)',
    '3723255316532': 'Maxlite 5 Slim Expandable Cabin Soft Shell Spinner 55cm (55 x 40 x 20 cm)',
    '3723257479220': 'Maxlite 5 Large Expandable Soft Shell Spinner 79cm (79 x  53 x 33 cm)',
    '3723271405620': 'Platinum Elite Medium Expandable Soft Shell Spinner 71cm (71 x 47 x 30 cm)',
    '3723273797684': 'Platinum Elite Large Expandable Soft Shell Spinner 83cm (83 x 53 x 33 cm)',
    '3723276320820': 'Platinum Elite  Rolling Garment Bag (61 x 62 x 27 cm)',
    '3723278221364': 'Platinum Elite Business Backpack (44 x 40 x 21 cm)',
    '3723279794228': 'Platinum Elite Expandable Business Brief (33 x 40,6 x 21 cm)',
    '3723282120756': 'Platinum Elite Slim Expandable Cabin Soft Shell Spinner 55cm (55 x 40 x 20 cm)',
    '6593328971828': 'Travelpro Essentials Shoe Bags 2 Pack',
    '6593329037364': 'Travelpro Essentials Medium Expandable/Compressible Packing Cube',
    '6593329070132': 'Travelpro Essentials Large Expandable/Compressible Packing Cube',
    '6593329102900': 'Travelpro Essentials MaxAccess Cubes Deluxe Hanging Toiletry Organizer',
    '6593329135668': 'Travelpro Essentials Washable Laundry Bag',
    '6593329201204': 'Travelpro Essentials MaxAccess Cubes Toiletry Organizer',
    '6593341456436': 'FlightCrew 5 Horizontal Rolling Overnighter',
    '6593341489204': 'FlightCrew 5 22" Expandable Rollaboard',
    '6593341521972': 'FlightCrew 5 21" Rollaboard',
    '6593341554740': 'FlightCrew 5 Vertical Rolling Overnighter',
    '6593341751348': 'FlightCrew 5 Flight Tote',
    '6593341784116': 'FlightCrew 5 Multi-Purpose Tote',
    '6703222358068': 'Maxlite Air Large Expandable Hard Shell Spinner 78cm (78 x 49 x 30 cm)',
    '6703222390836': 'Maxlite Air Medium Expandable Hard Shell Spinner 70cm (70 x 44 x 28 cm)',
    '6703222456372': 'Maxlite Air Compact Expandable Cabin Hard Shell Spinner 55cm (55 x 35 x 23 cm)',
    '6707684048948': 'Maxlite Air Slim Cabin Hard Shell Spinner 55cm (55 x 40 x 20 cm)',
    '6933839478836': 'FlightCrew 5 22" Pilot Expandable Rollaboard',
    '6977197408308': 'Platinum Elite Medium Expandable Hard Shell Spinner 69cm (69 x 46 x 33cm)',
    '6977197473844': 'Platinum Elite Large Expandable Hard Shell Spinner 76cm (76 x 46 x 34cm)',
    '6977197506612': 'Platinum Elite Compact Expandable Cabin Hard Shell Spinner 55cm (55 x 35 x 23cm)',
    '7039773835316': 'Maxlite 5 Rolling Underseat Cabin Luggage',
    '8598492119365': 'Platinum Elite Expandable Cabin Hard Shell Spinner 55cm (55 x 40 x 20cm)',
    '8615919583557': 'Maxlite Laptop Backpack',
    '8683593007429': 'FlightCrew 5 21" Expandable Spinner',
    '8683593040197': 'FlightCrew 5 18" Expandable Rollaboard',
    '8683593072965': 'FlightCrew 5 22" Rollaboard',
    '8683593138501': 'FlightCrew 5 24" Expandable Rollaboard',
    '8683593498949': 'FlightCrew 5 Large Crew Cooler',
    '8683593531717': 'FlightCrew 5 Deluxe Tote',
    '8683593597253': 'FlightCrew 5 Crew Cooler',
    '8683593793861': 'FlightCrew 5 Slim Expandable Cabin Rollaboard',
    '8683593826629': 'FlightCrew 5 City Tote',
    '8683633541445': 'Maxlite Rolling Medium Garment Bag',
    '8683633606981': 'Maxlite 5 Compact Expandable Cabin Spinner',
    '8683633639749': 'Maxlite 5 International Expandable Cabin Rollaboard',
    '8705326580037': 'Maxlite 5 Drop-Bottom Weekender',
    '8815730196805': 'Crew Classic UnderSeat Tote',
    '8815730229573': 'Crew Classic Rolling UnderSeat Cabin Luggage',
    '8815730262341': 'Crew Classic Large Expandable Spinner',
    '8815730458949': 'Crew Classic Expandable Cabin Rollaboard',
    '8815730557253': 'Crew Classic Medium Expandable Spinner',
    '8815730590021': 'Crew Classic Compact Expandable Cabin Spinner',
    '9166105477445': 'Travelpro Essentials Waist Bag',
    '9166105510213': 'Travelpro Essentials SparePack Foldable Tote',
    '9166105542981': 'Travelpro Essentials Garment Packing Folder',
    '9166105575749': 'Travelpro Essentials XL Expandable/Compressible Packing Cube',
    '9166105739589': 'Travelpro Essentials Leather Passport Cover',
    '9166105805125': 'Travelpro x Travel + Leisure UnderSeat Tote',
    '9166105837893': 'Travelpro Essentials Leather Luggage Tag',
    '9166106034501': 'Travelpro x Travel + Leisure Slim Backpack',
    '9166106132805': 'Travelpro x Travel + Leisure Drop-Bottom Weekender',
    '9166106165573': "Maxlite Women's Tote",
    '9604803920197': 'VersaPack+ Large Spinner',
    '9604803952965': 'VersaPack+ Medium Spinner',
    '14977855717701': 'FC5 TOP & SIDE CARRY HANDLE',
    '9604805001541': 'VersaPack+ UnderSeat Tote',
    '9616227238213': 'Travelpro Pilot Expandable Cabin Rollaboard',
    '9632821117253': 'Platinum Elite UnderSeat Tote',
    '9632821150021': 'Platinum Elite Soft Duffel',
    '9632821182789': 'Platinum Elite Drop-Bottom Weekender',
    '9632821215557': 'Platinum Elite Business Backpack',
    '9632821346629': 'Platinum Elite Slim Backpack',
    '9658124468549': 'Travelpro Pilot Seven3 Cabin Rollaboard (no side pockets/expansion)',
    '14786384888133': 'Platinum Elite Slim Backpack - Camouflage',
    '14786384920901': 'Platinum Elite Drop-Bottom Weekender - Camouflage',
    '14809388843333': 'Travelpro Altitude Organisation Kit',
    '14809388908869': 'Travelpro Altitude Double Expansion Duffel',
    '14809388941637': 'Travelpro Altitude Full Expansion Brief',
    '14809389564229': 'Travelpro Altitude Large Expandable Laptop Backpack 30-36L',
    '14809389957445': 'Travelpro Altitude Medium Expandable Laptop Backpack 25-30L',
    '14809390022981': 'Travelpro Altitude Slim Expandable Laptop Backpack 20-24L',
    '14822103941445': 'Platinum Elite UnderSeat Tote - Camouflage',
    '14822103974213': 'Platinum Elite Soft Duffel - Camouflage',
    '14822104006981': 'Platinum Elite Business Backpack - Camouflage',
    '14977852408133': 'ML AIR SIDE FEET KIT - FOR C/I',
    '14977852506437': 'FC5 BOTTOM FOOT WITH HANDLE',
    '14977852539205': 'LEFT BOTTOM FEET ROLLABOARD',
    '14977852604741': 'PE HS CARRY HANDLE ASM KIT',
    '14977852670277': 'PE HS C/O TOP HANDLE ASM KIT',
    '14977852735813': 'PE HS MED&LG C/I TOP HNDL KIT',
    '14977852801349': 'LEFT WHEEL KIT MED/LRG SPIN',
    '14977852866885': 'PE HS LEFT DBL-WHEEL ASM KIT',
    '14977852965189': 'RIGHT WHEEL KIT MED/LRG SPIN',
    '14977853063493': 'PE HS RIGHT DBL-WHEEL ASM KIT',
    '14977853161797': 'ML AIR WHEELS HOUSING ASSY KIT',
    '14977853227333': 'LEFT WHEEL C/O RB UNDRSEAT C/O',
    '14977853292869': 'RGT WHEEL C/O RB UNDRSEAT C/O',
    '14977853358405': 'ML AIR 2 STG RETRACT HAND SYS',
    '14977853423941': 'FC5 3 STAGE RETRACT HANDLE SYS',
    '14977853456709': '3 STAGE HANDLE C/O EXP RB',
    '14977853522245': '3 STAGE RETRACT HS SYSTEM',
    '14977853587781': '3 STG RETRACT HANDLE COMP C/O',
    '14977853686085': '2 STAGE 24" HS EXP RB HANDLE',
    '14977853784389': '2 STG RETRACT HANDLE MED C/I',
    '14977853882693': '2 STG RETRACT HANDLE LRG C/I',
    '14977853948229': '5 STAGE HANDLE UNDERSEAT C/O',
    '14977854013765': '3 STAGE RETRACT HANDLE KIT',
    '14977854079301': 'FC5 WHEEL ASSEMBLY KIT',
    '14977854177605': 'ML AIR TSA COMBINATIN LOCK KIT',
    '14977854308677': 'USB HOUSING',
    '14977854374213': 'DIY M/F J-HOOK REPLACEMENT',
    '14977854406981': 'ML5  LEFT BOTTOM FOOT',
    '14977854472517': 'ML5  RIGHT BOTTOM FOOT',
    '14977854538053': 'ML5 5-STG RET HNDL RLLG TOTE',
    '14977854603589': 'ML5 5-STG RET HDLE RLLG DUFFLE',
    '14977854669125': 'ML5 5-STG RET HDLE UND SEAT',
    '14977854767429': 'ML5 2-STG RT HNDL 25IN EX SPNR',
    '14977854800197': 'ML5 3-STG RT HNDL 20IN EX SPNR',
    '14977854898501': 'ML5 3-STG RT HNDL INTL EX RLBD',
    '14977854996805': 'ML5 LEFT WHEEL HOUSNG W/WHEEL',
    '14977855029573': 'ML5 RIGHT WHEEL HOUSNG W/WHEEL',
    '14977855127877': 'ML5 RG/REAR LFT/FRONT SPINNER',
    '14977855816005': 'FC5 FRNT LEFT/BACK RGT SPR WHL',
    '14977855586629': 'ML5#10 LCKNG SLDR LOGO BLK',
    '9604804018501': 'VersaPack+ Compact  Expandable Cabin Spinner',
}

if __name__ == "__main__":
  # Set the file name to save the reviews under the current date
  today = date.today()
  fName = f"TravelPro-EU-Reviews-{today}.txt"
  file = open(fName, "w")
  file.write("------------------------------------------------------------------\n")
  file.write("Travelpro EU store reviews (eu.travelpro.com)\n")
  file.write("------------------------------------------------------------------\n\n")

  # Loop through each product ID in the PRODUCTS dictionary
  # and fetch the reviews for each product.
  for key in PRODUCTS:
    reviews = fetch_all_reviews(APP_KEY, key)
    file.write(f"PRODUCT NAME: {PRODUCTS[key]}\n")
    file.write(f"Total reviews: {len(reviews)}\n\n")
    for i in range(len(reviews)):
      print_review(i, reviews, file, PRODUCTS[key])

  # Remember to close the file after writing to it.
  file.close()